# Quick Colab Evaluation - CustomCNN v2

This notebook is a lightweight course-submission demo. It does **not** retrain the model. It loads the saved `CustomCNN v2` checkpoint and evaluates it on a 100-image subset copied from the fixed `split_v1` test split.

Expected folder layout after uploading to Colab:

```text
/content/colab_quick_eval/
|-- customcnn_v2_colab_eval.ipynb
|-- test_manifest.csv
|-- test_images/
|   |-- cats/
|   |-- dogs/
|   `-- wildlife/
`-- weights/
    |-- customcnn_v2_checkpoint.pt
    |-- customcnn_v2_config.json
    `-- customcnn_v2_metrics.json
```

If the folder is uploaded somewhere else, update `PROJECT_DIR` in the first code cell.

In [1]:
from pathlib import Path
import zipfile
import shutil

ZIP_PATH = Path("/content/colab_quick_eval_fixed.zip")

if not ZIP_PATH.exists():
    ZIP_PATH = Path("/content/colab_quick_eval.zip")

if not ZIP_PATH.exists():
    raise FileNotFoundError("Upload colab_quick_eval_fixed.zip or colab_quick_eval.zip to /content first.")

EXTRACT_ROOT = Path("/content")
PROJECT_DIR = EXTRACT_ROOT / "colab_quick_eval"

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_ROOT)

# Fix case where zip extracts as /content/colab_quick_eval/colab_quick_eval
nested = PROJECT_DIR / "colab_quick_eval"
if nested.exists():
    tmp = EXTRACT_ROOT / "_colab_quick_eval_tmp"
    if tmp.exists():
        shutil.rmtree(tmp)
    nested.rename(tmp)
    shutil.rmtree(PROJECT_DIR)
    tmp.rename(PROJECT_DIR)

print("Extracted to:", PROJECT_DIR)
print("Images:", len(list((PROJECT_DIR / "test_images").rglob("*.*"))))
print("Weights:", list((PROJECT_DIR / "weights").iterdir()))


Extracted to: /content/colab_quick_eval
Images: 100
Weights: [PosixPath('/content/colab_quick_eval/weights/customcnn_v2_metrics.json'), PosixPath('/content/colab_quick_eval/weights/customcnn_v2_config.json'), PosixPath('/content/colab_quick_eval/weights/customcnn_v2_checkpoint.pt')]


In [2]:
from pathlib import Path
import json
import time

PROJECT_DIR = Path('/content/colab_quick_eval')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

TEST_IMAGES_DIR = PROJECT_DIR / 'test_images'
CHECKPOINT_PATH = PROJECT_DIR / 'weights' / 'customcnn_v2_checkpoint.pt'
CONFIG_PATH = PROJECT_DIR / 'weights' / 'customcnn_v2_config.json'

print('Project dir:', PROJECT_DIR)
print('Test images:', TEST_IMAGES_DIR, TEST_IMAGES_DIR.exists())
print('Checkpoint:', CHECKPOINT_PATH, CHECKPOINT_PATH.exists())

if not TEST_IMAGES_DIR.exists():
    raise FileNotFoundError(f'Missing test_images directory: {TEST_IMAGES_DIR}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing checkpoint: {CHECKPOINT_PATH}')

Project dir: /content/colab_quick_eval
Test images: /content/colab_quick_eval/test_images True
Checkpoint: /content/colab_quick_eval/weights/customcnn_v2_checkpoint.pt True


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


In [4]:
class CustomCNNv2(nn.Module):
    """Same architecture as src.models.cnn_scratch.models.CustomCNNv2."""

    def __init__(self, num_classes=3, input_channels=3, dropout_p=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def extract_state_dict(checkpoint):
    """Accept common checkpoint formats used by PyTorch training utilities."""
    if isinstance(checkpoint, dict):
        for key in ['model_state_dict', 'state_dict', 'model_state', 'best_model_state_dict', 'best_state_dict']:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                return checkpoint[key]
    return checkpoint

model = CustomCNNv2(num_classes=3, input_channels=3, dropout_p=0.5).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = extract_state_dict(checkpoint)
state_dict = {k.replace('module.', '', 1): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()
print('Loaded CustomCNN v2 checkpoint successfully.')

Loaded CustomCNN v2 checkpoint successfully.


In [5]:
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(TEST_IMAGES_DIR, transform=eval_transform)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

print('Class mapping:', dataset.class_to_idx)
print('Images:', len(dataset))
if len(dataset) != 100:
    print('Warning: expected 100 images, found', len(dataset))

Class mapping: {'cats': 0, 'dogs': 1, 'wildlife': 2}
Images: 100


In [6]:
y_true = []
y_pred = []
start = time.perf_counter()

with torch.no_grad():
    for images, labels in loader:
        images = images.to(DEVICE)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu()
        y_true.extend(labels.tolist())
        y_pred.extend(preds.tolist())

elapsed = time.perf_counter() - start
idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')
cm = confusion_matrix(y_true, y_pred)

print(f'Accuracy: {acc:.4f}')
print(f'Macro F1: {macro_f1:.4f}')
print(f'Evaluation time: {elapsed:.3f} sec')
print(f'Images/sec: {len(dataset) / elapsed:.2f}')
print('\nConfusion matrix rows=true, cols=pred:')
print(class_names)
print(cm)
print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

Accuracy: 0.9400
Macro F1: 0.9400
Evaluation time: 1.585 sec
Images/sec: 63.08

Confusion matrix rows=true, cols=pred:
['cats', 'dogs', 'wildlife']
[[31  1  2]
 [ 1 32  0]
 [ 1  1 31]]

Classification report:
              precision    recall  f1-score   support

        cats     0.9394    0.9118    0.9254        34
        dogs     0.9412    0.9697    0.9552        33
    wildlife     0.9394    0.9394    0.9394        33

    accuracy                         0.9400       100
   macro avg     0.9400    0.9403    0.9400       100
weighted avg     0.9400    0.9400    0.9399       100

